In [2]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf

# Load model + tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Try setting use_safetensors=False to force loading a legacy format (like a H5 file)
model = TFAutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    use_safetensors=False
)

# Sample data
texts = ["I loved this movie!", "This film was terrible."]
labels = [1, 0]

# Tokenize
enc = tokenizer(texts, padding=True, truncation=True, return_tensors="tf")

# Train



TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_layer_norm', 'vocab_transform', 'activation_13', 'vocab_projector']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-

In [3]:
model.compile(
    # Keep the optimizer
    optimizer=tf.keras.optimizers.Adam(2e-5),
    
    # REMOVE the 'loss' argument entirely. 
    # The model will compute loss internally from the labels passed to model.fit().
    
    # Keep the metrics
    metrics=["accuracy"]
)

model.fit(enc.data, tf.constant(labels), epochs=2)

# Predict
out = model(enc.data)
print(out.logits)

Epoch 1/2


1/1 [==============================] - 28s 28s/step - loss: 0.7047 - accuracy: 0.5000
Epoch 2/2
1/1 [==============================] - 1s 1s/step - loss: 0.7167 - accuracy: 0.5000
tf.Tensor(
[[ 0.0172319   0.01262921]
 [ 0.05889345 -0.01114279]], shape=(2, 2), dtype=float32)


In [4]:
from datasets import load_dataset

# Loads the IMDb dataset
imdb_dataset = load_dataset("imdb")

# Access the training split
train_data = imdb_dataset["train"]

print(f"Number of training examples: {len(train_data)}")
# Output: Number of training examples: 25000

Number of training examples: 25000


In [5]:
# Extract the raw texts and labels
raw_texts = train_data["text"]
labels = train_data["label"] 

# Use a small subset for quick testing (25000 is still quite large)
sample_texts = raw_texts[:2000]
sample_labels = labels[:2000]

# --- Apply your tokenizer ---
# Assuming 'tokenizer' and 'tf' are defined from your previous code


# Now, you can run your model.fit() command:
# model.fit(enc.data, tf_labels, epochs=2)

In [ ]:
from transformers import AutoTokenizer
import tensorflow as tf
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

enc = tokenizer(sample_texts, padding=True, truncation=True, return_tensors="tf" )

tf_labels = tf.constant(sample_labels)

In [ ]:
model.compile( optimizer=tf.keras.optimizers.Adam(2e-5), metrics=["accuracy"] ) 
model.fit(enc.data, tf_labels, epochs=20)

Epoch 1/20
 2/63 [..............................] - ETA: 6:06:22 - loss: 0.6394 - accuracy: 0.8594

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# 1. Choose a pre-trained model
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Load your dataset (example: IMDB)
dataset = load_dataset("imdb")

# 3. Tokenize
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_ds = dataset.map(preprocess, batched=True)

# 4. Training config
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 5. Train it
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"]
)

trainer.train()

# 6. Predict
text = "I loved the direction and storyline"
inputs = tokenizer(text, return_tensors="pt")
logits = model(**inputs).logits
prediction = logits.argmax(dim=1).item()
print(prediction)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\aswin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aswin\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

dataset = load_dataset("imdb")

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_ds = dataset.map(preprocess, batched=True)

# Training args without evaluation_strategy
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,   # keep logs
    # no evaluation_strategy here
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    # you can still pass eval_dataset but automatic eval may not run depending on transformers version
    eval_dataset=tokenized_ds["test"],
)

trainer.train()

# Manual evaluation step (works regardless of TrainingArguments API)
metrics = trainer.evaluate(tokenized_ds["test"])
print(metrics)


: 

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# 1. Choose a pre-trained model
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Load your dataset (example: IMDB)
dataset = load_dataset("imdb")

# 3. Tokenize
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_ds = dataset.map(preprocess, batched=True)

# 4. Training config
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 5. Train it
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"]
)

trainer.train()

# 6. Predict
text = "I loved the direction and storyline"
inputs = tokenizer(text, return_tensors="pt")
logits = model(**inputs).logits
prediction = logits.argmax(dim=1).item()
print(prediction)


: 

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# 1. Choose a pre-trained model
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Load your dataset (example: IMDB)
dataset = load_dataset("imdb")

# 3. Tokenize
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_ds = dataset.map(preprocess, batched=True)


: 

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
    disable_tqdm=False,
)


In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# 1. Load dataset (example: IMDB)
dataset = load_dataset("imdb")

# 2. Load model + tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

# 3. Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")

# 4. Training settings
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_steps=20,
    report_to="none"
)

# 5. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(2000)),  # sample for speed
)

# 6. Train
trainer.train()


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Step,Training Loss


KeyboardInterrupt: 